In [88]:
from pathlib import Path
import os

# Move from /notebooks to project root
ROOT = Path(os.getcwd()).parent

# Folder where your PDFs are
PDF_FOLDER = ROOT / "data" / "raw"

# Find ALL pdfs recursively inside /data/raw/
pdf_files = list(PDF_FOLDER.rglob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(f"No PDF files found inside {PDF_FOLDER}")

print(f"Found {len(pdf_files)} PDF files:")
for f in pdf_files:
    print(" -", f)


Found 1 PDF files:
 - c:\Users\maumo\OneDrive\Dokumente\rag_project\data\raw\din_5566_2.pdf


In [89]:
from langchain_community.document_loaders import PyPDFLoader # Langchain PDF Loader change if necessary
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

def text_formatter(text: str) -> str:
    """Format text function to clean up extracted text."""

    # Add if neccessary more formatting steps
    # Getting rid of multiple new lines
    text = re.sub(r"\n+", " ", text)
    
    return text.strip()

def read_pdf(pdf_path: str | Path):
    """Read PDF and return list of cleaned pages."""
    pdf_path = Path(pdf_path)
    loader = PyPDFLoader(str(pdf_path))

    raw_docs = loader.load()   # List[Document]

    cleaned_docs = []

    for d in raw_docs:
        page_number = d.metadata.get("page", None)

        cleaned_text = text_formatter(d.page_content)

        new_doc = Document(
            page_content=cleaned_text,
            metadata={
                "source": pdf_path.name,          # filename
                "filepath": str(pdf_path),        # full path
                "page": page_number + 1,              # page number
                #"doctype": "norm",                # optional
                "doc_id": pdf_path.stem.lower(),  # e.g. "din_5566_2"
            }
        )

        cleaned_docs.append(new_doc)

    return cleaned_docs
docs = read_pdf(str(PDF_PATH))
#print(docs[5])  # erstes Document Objekt anzeigen


In [ ]:
#Multi docs use case
all_docs = []
for pdf_path in pdf_files:
    docs = read_pdf(pdf_path)
    all_docs.extend(docs)


In [81]:
#Chunking the documents
def chunk_documents(docs: list[Document], chunk_size: int = 1000, chunk_overlap: int = 200):
    
    """Split cleaned documents into smaller pieces while preserving metadata."""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,                       #The metadata will be passed with langchain to the chunks 
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    chunks = text_splitter.split_documents(docs)
    return chunks


In [ ]:
#Testing spot for chunking
chunked_docs = chunk_documents(docs)
print(f"Number of chunks created: {len(chunked_docs)}")
print(chunked_docs[8].metadata)

Number of chunks created: 53
{'source': 'din_5566_2.pdf', 'filepath': 'c:\\Users\\maumo\\OneDrive\\Dokumente\\rag_project\\data\\raw\\din_5566_2.pdf', 'page': 2, 'doc_id': 'din_5566_2'}


In [ ]:
import pandas as pd
df = pd.DataFrame(docs)
df.head()

ModuleNotFoundError: No module named 'langchain_community.emmbeddings'

In [48]:
import re

DIN_HEADER_PATTERNS = [
    r"DEUTSCHE\s*NORM.*",
    r"Alleinverkauf.*Beuth\s+Verlag.*",
    r"Printed\s*copies\s*are\s*uncontrolled.*",
    r"Datum\s*/\s*Uhrzeit\s*des\s*Ausdrucks:.*",
    r"www\.din\.de|www\.beuth\.de",
    r"ICS\s*\d{2}\.\d{3}\.\d{2}",
    r"©.*",
]

def clean_din_text(text: str) -> str:
    # 1) Trennstriche über Zeilen umbrechen
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    # 2) Kopf-/Fußzeilen raus
    for pat in DIN_HEADER_PATTERNS:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    # 3) Zeilenumbrüche normalisieren
    text = re.sub(r"[ \t]*\n[ \t]*", "\n", text)        # Zeilen trimmen
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)        # einzelne \n → Leerzeichen
    text = re.sub(r"\n{3,}", "\n\n", text)              # >2 \n → 2 \n

    # 4) Mehrere Spaces zusammenfassen
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

loader = PyPDFLoader(str(PDF_PATH))
docs = loader.load()

print("Seiten vor Cleaning:", len(docs))
print("Erste 500 Zeichen vorher:\n")
print(docs[12].page_content[:500])



Seiten vor Cleaning: 17
Erste 500 Zeichen vorher:

DIN 5566-2:2020-05 
13 
Anhang A 
(normativ) 
 
Sichtbedingungen (schematisch) 
Maße in Millimeter 
 
Legende 
1 Augenposition größte Person, stehend, nach DIN 5566-1:2020-05, Tabelle 1 
2 Augenposition größte Person, sitzend, nach DIN 5566-1:2020-05, Tabelle 1 
3 Augenposition kleinste Person, sitzend, nach DIN 5566-1:2020-05, Tabelle 1 
4 Sicht auf hohe Signale (größte Person, stehend) bei Frontscheibenoberkante 1 950 mm über 
Fußbodenoberkante 
5 Sicht auf hohe Signale (größte Person, sitzend


In [52]:
for d in docs:
    d.page_content = clean_din_text(d.page_content)

print("Erste 500 Zeichen nach Cleaning:\n")
print(docs[4].page_content[:500])


Erste 500 Zeichen nach Cleaning:

DIN 5566-2:2020-05 5 4 Plätze für das Personal 4.1 Allgemeines Die Führerräume müssen so aus geführt werden , dass Einmannbedienung sicherges tellt ist und dass der Eisenbahnfahrzeugführer das Fahrzeug sowohl im Sitzen als auch im Stehen führen kann. Bei besonderen Einsatzbedingungen kann davon abgewichen werden. Weiterhin muss im Führerraum eine Begleitersitzgelegenheit vor gesehen werden, von d er aus die Beobachtung der Strecke möglich sein muss. 4.2 Anordnung der Plätze Der Platz des Eisenba
